In [1]:
import pathlib

import cv2
import pandas
from metavision_sdk_base import EventCD, EventCDBuffer
from metavision_sdk_core import PeriodicFrameGenerationAlgorithm, BaseFrameGenerationAlgorithm
from metavision_sdk_ui import EventLoop, BaseWindow, MTWindow, UIAction, UIKeyEvent

import numpy as np

from tqdm.autonotebook import tqdm

C:\Users\Bas_K\AppData\Local\Temp\ipykernel_31648\1887005993.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
dir_path = R"C:\Users\Bas_K\source\repos\Thesis\thesis-insect-monitoring\datasets\insect_combined\full_trajectories"
events_chunksize = 5000
width = 1280
height = 720

classes = {
    "insect": 0,
    "ins": 0,
    "bee": 1,
    "bumblebee": 2,
    "bum": 2,
    "butterfly": 3,
    "but": 3,
    "dragonfly": 4,
    "dra": 4,
    "wasp": 5,
    "was": 5,
}

dir_path = pathlib.Path(dir_path)

dirs = []
for path in dir_path.iterdir():
    if path.is_file():
        continue
    dirs.append(path)

In [3]:
def process_dir(dir_path):
    dfs = []
    for file in dir_path.iterdir():
        if file.is_file() and file.suffix == ".csv" and not file.stem.startswith("combined"):
            df = pandas.read_csv(file)
            df["source"] = file.stem
            df["t"] += int(file.stem.split("_")[-1].split("start")[-1])
            dfs.append(df)

    return pandas.concat(dfs)

In [4]:
def callback(ts, frame, df, output_path, state):
    cv2.imwrite(str(output_path / f"frame_{ts}.png"), frame)

    chunk = df[(df["t"] >= state["prev_ts"]) & (df["t"] < ts)]

    min_x = chunk['x'].min() / width
    max_x = chunk['x'].max() / width
    min_y = chunk['y'].min() / height
    max_y = chunk['y'].max() / height
    id = classes[output_path.parent.stem.split("_")[-1]]

    pandas.DataFrame([[id, (min_x + max_x) / 2, (min_y + max_y) / 2, max_x - min_x, max_y - min_y]]).to_csv(output_path / f"frame_{ts}.csv", index=False, header=False)

    state["prev_ts"] = ts

In [7]:
accumulation_times_us = [5000, 10000, 33333]

for accumulation_time_us in tqdm(accumulation_times_us, desc="Accumulation times"):
    for directory in tqdm(dirs, desc=f"Dirs ({accumulation_time_us} us)", leave=False):
        df = process_dir(directory)
        df.sort_values(by=["t"]).drop(columns=["source"]).to_csv(directory / "combined.csv", index=False)
        sources = df["source"].unique()

        for source in tqdm(sources, desc="Sources", leave=False):
            state = {"prev_ts": 0}
            output_path = directory / "_".join(source.split("_")[:2]) / f"acc_time_{accumulation_time_us}"
            output_path.mkdir(parents=True, exist_ok=True)

            df_filtered = df[df["source"] == source].sort_values(by=["t"]).drop(columns=["source"])
            buf = df_filtered.to_numpy()

            event_frame_gen = PeriodicFrameGenerationAlgorithm(width, height, accumulation_time_us)
            event_frame_gen.set_output_callback(
                lambda ts, frame: callback(ts, frame, df_filtered, output_path, state))

            for i in tqdm(
                    range(0, len(buf), events_chunksize),
                    desc=f"Chunks of {source}",
                    leave=False
            ):
                chunk = buf[i:i + events_chunksize]
                events_buf = EventCDBuffer(len(chunk))
                np_evs = events_buf.numpy()

                np_evs['x'] = chunk[:, 0]
                np_evs['y'] = chunk[:, 1]
                np_evs['t'] = chunk[:, 2]
                np_evs['p'] = chunk[:, 3]

                event_frame_gen.process_events(np_evs)

        # output_path = directory / "combined" / f"acc_time_{accumulation_time_us}"
        # output_path.mkdir(parents=True, exist_ok=True)
        #
        # df_filtered = df.drop(columns=["source"])
        # buf = df_filtered.to_numpy()
        #
        # event_frame_gen = PeriodicFrameGenerationAlgorithm(width, height, accumulation_time_us)
        # event_frame_gen.set_output_callback(
        #     lambda ts, frame: cv2.imwrite(str(output_path / f"frame_{ts}.png"), frame))
        #
        # for i in range (0, len(buf), events_chunksize):
        #     chunk = buf[i:i + events_chunksize]
        #     events_buf = EventCDBuffer(len(chunk))
        #     np_evs = events_buf.numpy()
        #
        #     np_evs['x'] = chunk[:, 0]
        #     np_evs['y'] = chunk[:, 1]
        #     np_evs['t'] = chunk[:, 2]
        #     np_evs['p'] = chunk[:, 3]
        #
        #     event_frame_gen.process_events(np_evs)





Chunks of 9_bee_pts686813_start20972116:   0%|          | 0/138 [00:00<?, ?it/s]

In [5]:
# accumulation time of 5000 resulted in 718 for hn-bee-1/0_bee, however, many frames had a sparse number of events. So the aim is to have less frames with more events per frame
# 10 Events per frame resulted in 2279 frames, too much.
# 50 Events per frame resulted in 455 frames, a lot less however the amount of events is still on the low side.
# 100 Events per frame resulted in 227 frames, seemed to be a good event count. When going higher then this objects start to create trails behind them when moving fast.

event_windows = [100]

for event_window in tqdm(event_windows, desc="Accumulation times"):
    for directory in tqdm(dirs, desc=f"Dirs ({event_window} events)", leave=False):
        df = process_dir(directory)
        df.sort_values(by=["t"]).drop(columns=["source"]).to_csv(directory / "combined.csv", index=False)
        sources = df["source"].unique()

        for source in tqdm(sources, desc="Sources", leave=False):
            state = {"prev_ts": 0}
            output_path = directory / "_".join(source.split("_")[:2]) / f"evpf_{event_window}"
            output_path.mkdir(parents=True, exist_ok=True)

            df_filtered = df[df["source"] == source].sort_values(by=["t"]).drop(columns=["source"])
            buf = df_filtered.to_numpy()

            for i in tqdm(
                    range(0, len(buf), event_window),
                    desc=f"Frames of {source}",
                    leave=False
            ):
                chunk = buf[i:i + event_window]
                # Skip last chunk if it has fewer events than the event_window.
                if len(chunk) < event_window:
                    continue

                events_buf = EventCDBuffer(len(chunk))
                np_evs = events_buf.numpy()

                np_evs['x'] = chunk[:, 0]
                np_evs['y'] = chunk[:, 1]
                np_evs['t'] = chunk[:, 2]
                np_evs['p'] = chunk[:, 3]

                frame = np.zeros((height, width, 3), dtype=np.uint8)
                BaseFrameGenerationAlgorithm.generate_frame(np_evs, frame)

                ts = int(np_evs["t"][-1])
                callback(ts, frame, df_filtered, output_path, state)



Frames of 9_bee_pts686813_start20972116:   0%|          | 0/6869 [00:00<?, ?it/s]

In [18]:
import cv2

def draw_bbox_on_frame(frame, bbox_row, normalized=True, color=(0, 255, 0), thickness=2):
    """
    bbox_row needs: x_center, y_center, width, height
    If normalized=True, values are in [0,1] (YOLO style).
    """
    h, w = frame.shape[:2]

    xc = float(bbox_row["x_center"])
    yc = float(bbox_row["y_center"])
    bw = float(bbox_row["width"])
    bh = float(bbox_row["height"])

    if normalized:
        xc *= w
        yc *= h
        bw *= w
        bh *= h

    x1 = int(round(xc - bw / 2))
    y1 = int(round(yc - bh / 2))
    x2 = int(round(xc + bw / 2))
    y2 = int(round(yc + bh / 2))

    # Clamp to image bounds
    x1 = max(0, min(x1, w - 1))
    y1 = max(0, min(y1, h - 1))
    x2 = max(0, min(x2, w - 1))
    y2 = max(0, min(y2, h - 1))

    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

    if "id" in bbox_row:
        cv2.putText(frame, str(bbox_row["id"]), (x1, max(0, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

    return frame


In [19]:
img = cv2.imread(R"C:\Users\Bas_K\source\repos\Thesis\thesis-insect-monitoring\datasets\insect_combined\full_trajectories\hn-bee-1\0_bee\evpf_100\frame_29890.png")
df = pandas.read_csv(R"C:\Users\Bas_K\source\repos\Thesis\thesis-insect-monitoring\datasets\insect_combined\full_trajectories\hn-bee-1\0_bee\evpf_100\frame_29890.csv", names=["id", "x_center", "y_center", "width", "height"])

In [20]:
frame = draw_bbox_on_frame(img, df)

C:\Users\Bas_K\AppData\Local\Temp\ipykernel_39788\1255065631.py:10: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  xc = float(bbox_row["x_center"])
C:\Users\Bas_K\AppData\Local\Temp\ipykernel_39788\1255065631.py:11: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  yc = float(bbox_row["y_center"])
C:\Users\Bas_K\AppData\Local\Temp\ipykernel_39788\1255065631.py:12: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  bw = float(bbox_row["width"])
C:\Users\Bas_K\AppData\Local\Temp\ipykernel_39788\1255065631.py:13: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  bh = float(bbox_row["height"])


In [21]:
cv2.imshow("frame", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()